In [ ]:
import matplotlib.pyplot as plt
plt.style.use("utils/RESS.mplstyle")

# Figure 11 
Comparision of the mono and bi-level approach  

Loading and postprocessing the bi-level results

In [ ]:
from pathlib import Path
import csv
import re
import torch

def load_refererence(name):
    if name == "Hartmann6":
        ref_data = torch.load("_reference_comp/Hartmann6_uq_tau0p95.pt", weights_only=False)
    elif name == "Sab2d":
        ref_data = torch.load("_reference_comp/Sab2d_quad_uq_tau0p95.pt", weights_only=False)
    elif name == "BraninHoo":
        ref_data = torch.load("_reference_comp/BraninHooUQ_nonlin_uq_tau0p95.pt", weights_only=False)
    x_star, q_star = ref_data.x_star, ref_data.q_star
    return x_star, q_star

def find_optimization_files(folder, query):
    """
    Return all CSV files whose names start with 'optimization_results_'
    and contain the query, ignoring case, spaces, and punctuation.
    """
    folder = Path(folder)

    def normalize(text):
        return re.sub(r"[^a-z0-9]", "", text.lower())

    normalized_query = normalize(query)

    return [
        file
        for file in folder.iterdir()
        if (
            file.is_file()
            and file.suffix.lower() == ".csv"
            and file.name.lower().startswith("optimization_results_")
            and normalized_query in normalize(file.stem)
        )
    ]

def extract_tensors(file_path, dtype=torch.float32):
    file_path = Path(file_path)

    with file_path.open("r", newline="") as file:
        reader = csv.DictReader(file, delimiter="\t")

        fieldnames = [name.strip() for name in reader.fieldnames]
        reader.fieldnames = fieldnames

        x_columns = sorted(
            [
                name
                for name in fieldnames
                if re.fullmatch(r"x\d+", name)
            ],
            key=lambda name: int(name[1:])
        )

        expected_x_columns = [
            f"x{i}" for i in range(len(x_columns))
        ]

        if x_columns != expected_x_columns:
            raise ValueError(
                f"Expected consecutive x columns starting at x0, "
                f"but found: {x_columns}"
            )

        required_columns = [
            "value",
            "total_iterations",
            *x_columns,
        ]

        missing_columns = [
            column
            for column in required_columns
            if column not in fieldnames
        ]

        if missing_columns:
            raise ValueError(
                f"Missing columns: {missing_columns}"
            )

        values = []
        x_data = []
        total_iterations = []

        for row in reader:
            row = {
                key.strip(): value.strip()
                for key, value in row.items()
            }

            values.append(float(row["value"]))

            x_data.append([
                float(row[column])
                for column in x_columns
            ])

            total_iterations.append(
                float(row["total_iterations"])
            )

    values_tensor = torch.tensor(values, dtype=dtype)

    x_tensor = torch.tensor(x_data, dtype=dtype)

    total_iterations_tensor = torch.tensor(
        total_iterations,
        dtype=dtype,
    )

    return values_tensor, x_tensor, total_iterations_tensor

import torch


def get_best_so_far(q, x):
    if q.ndim != 1:
        raise ValueError("q must be a one-dimensional tensor")

    if x.ndim != 2:
        raise ValueError("x must be a two-dimensional tensor")

    if q.shape[0] != x.shape[0]:
        raise ValueError("q and x must have the same number of rows")

    q_best, best_indices = torch.cummin(q, dim=0)
    x_best = x[best_indices]

    return q_best, x_best

def compute_relative_errors(x_star, q_star, q_best, x_best):
    if x_star.ndim != 1:
        raise ValueError("x_star must be a one-dimensional tensor")

    if q_star.numel() != 1:
        raise ValueError("q_star must contain exactly one value")

    if q_best.ndim != 1:
        raise ValueError("q_best must be a one-dimensional tensor")

    if x_best.ndim != 2:
        raise ValueError("x_best must be a two-dimensional tensor")

    if q_best.shape[0] != x_best.shape[0]:
        raise ValueError(
            "q_best and x_best must have the same number of rows"
        )

    if x_best.shape[1] != x_star.shape[0]:
        raise ValueError(
            "x_best and x_star must have the same number of dimensions"
        )

    q_star_value = q_star.reshape(-1)[0]

    if q_star_value == 0:
        raise ValueError(
            "Relative error for q is undefined when q_star is zero"
        )

    x_star_norm = torch.linalg.vector_norm(x_star)

    if x_star_norm == 0:
        raise ValueError(
            "Relative error for x is undefined when x_star has zero norm"
        )

    q_rel_error = (
        torch.abs(q_best - q_star_value)
        / torch.abs(q_star_value)
    )

    x_rel_error = (
        torch.linalg.vector_norm(x_best - x_star, dim=1)
        / x_star_norm
    )

    return x_rel_error, q_rel_error

def align_error_trajectories(
    list_total_iter,
    list_x_rel_err,
    list_q_rel_err,
):
    if not (
        len(list_total_iter)
        == len(list_x_rel_err)
        == len(list_q_rel_err)
    ):
        raise ValueError(
            "All input lists must contain the same number of runs"
        )

    if len(list_total_iter) == 0:
        raise ValueError("At least one run is required")

    for total_iter, x_rel_err, q_rel_err in zip(
        list_total_iter,
        list_x_rel_err,
        list_q_rel_err,
    ):
        if total_iter.ndim != 1:
            raise ValueError(
                "Each total iteration tensor must be one-dimensional"
            )

        if x_rel_err.ndim != 1 or q_rel_err.ndim != 1:
            raise ValueError(
                "Each error tensor must be one-dimensional"
            )

        if not (
            len(total_iter)
            == len(x_rel_err)
            == len(q_rel_err)
        ):
            raise ValueError(
                "Each run must have matching tensor lengths"
            )

        if len(total_iter) == 0:
            raise ValueError("Runs cannot be empty")

        if torch.any(total_iter[1:] <= total_iter[:-1]):
            raise ValueError(
                "Total iterations must be strictly increasing"
            )

    common_total_iter = torch.unique(
        torch.cat(list_total_iter)
    ).sort().values

    aligned_x_rel_err = []
    aligned_q_rel_err = []

    for total_iter, x_rel_err, q_rel_err in zip(
        list_total_iter,
        list_x_rel_err,
        list_q_rel_err,
    ):
        indices = torch.searchsorted(
            total_iter,
            common_total_iter,
            right=True,
        ) - 1

        indices = indices.clamp_min(0)

        aligned_x_rel_err.append(x_rel_err[indices])
        aligned_q_rel_err.append(q_rel_err[indices])

    return (
        common_total_iter,
        torch.stack(aligned_x_rel_err),
        torch.stack(aligned_q_rel_err),
    )

def get_rel_error_bi_level(path, name):
    x_star, q_star = load_refererence(name = name)

    files = find_optimization_files(
        path,
        name
    )

    list_total_iter = []
    list_x_rel_err = []
    list_q_rel_err = []

    for file in files:
        print(file)
        values, x, total_iterations = extract_tensors(
            file
        )
        list_total_iter.append(total_iterations)
        
        q_best, x_best = get_best_so_far(values, x)

        x_rel_error, q_rel_error = compute_relative_errors(
            x_star,
            q_star,
            q_best,
            x_best,
        )

        list_x_rel_err.append(x_rel_error)
        list_q_rel_err.append(q_rel_error)

    (
        common_total_iter,
        x_rel_err_aligned,
        q_rel_err_aligned,
    ) = align_error_trajectories(
        list_total_iter,
        list_x_rel_err,
        list_q_rel_err,
    )

    total_iterations = total_iterations.detach().cpu()

    q_median = torch.quantile(q_rel_err_aligned, 0.5, dim=0, interpolation="nearest")
    q_lower = torch.quantile(q_rel_err_aligned, 0.1, dim=0, interpolation="nearest")
    q_upper = torch.quantile(q_rel_err_aligned, 0.9, dim=0, interpolation="nearest")

    x_median = torch.quantile(x_rel_err_aligned, 0.5, dim=0, interpolation="nearest")
    x_lower = torch.quantile(x_rel_err_aligned, 0.1, dim=0, interpolation="nearest")
    x_upper = torch.quantile(x_rel_err_aligned, 0.9, dim=0, interpolation="nearest")

    return common_total_iter, q_median, x_median, q_lower, q_upper, x_lower, x_upper

### Helper functioins - Mono-level Opt

In [ ]:
from notebooks.helper import get_batch_sim

eps = 1e-12

def get_mono_level_results(path, name):

    (
        best_q, best_x, model_calls_cumsum,
        n_EI, n_EM, n_TR, ref_x, ref_q,
    ) = get_batch_sim(path, name)

    best_q_rel = (
        torch.abs(best_q - ref_q) / torch.abs(ref_q).clamp_min(eps)
    ).clamp_min(eps)

    true_norm = torch.linalg.norm(ref_x).clamp_min(eps)
    best_x_rel = (
        torch.linalg.norm(best_x - ref_x, dim=-1) / true_norm
    ).clamp_min(eps)

    mono_x = model_calls_cumsum
    mono_q_median = torch.quantile(best_q_rel, 0.5, dim=1, interpolation="nearest")
    mono_q_lower = torch.quantile(best_q_rel, 0.1, dim=1, interpolation="nearest")
    mono_q_upper = torch.quantile(best_q_rel, 0.9, dim=1, interpolation="nearest")
    mono_x_median = torch.quantile(best_x_rel, 0.5, dim=1, interpolation="nearest")
    mono_x_lower = torch.quantile(best_x_rel, 0.1, dim=1, interpolation="nearest")
    mono_x_upper = torch.quantile(best_x_rel, 0.9, dim=1, interpolation="nearest")

    return mono_x, mono_q_median, mono_x_median, mono_q_lower, mono_q_upper, mono_x_lower, mono_x_upper

In [ ]:
from pathlib import Path

import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
ccycle = plt.rcParams['axes.prop_cycle'].by_key()['color']

from notebooks.helper import get_batch_sim

ccycle = plt.rcParams["axes.prop_cycle"].by_key()["color"]
eps = 1e-12

# Update paths/extensions to match your files.
problems = [
    {
        "title": r"Sabater-$g_2$",
        "mono_folder": "_experiments/RESS/sabater_batch_run_paper",
        "mono_name": "quad_uq",
        "bi_folder": "_experiments/RESS/Bi-level_paper_results",
        "bi_name": "Sab2d",
    },
    {
        "title": "BraninHoo",
        "mono_folder": "_experiments/RESS/branin_batch_run_paper",
        "mono_name": "nonlin_uq",
        "bi_folder": "_experiments/RESS/Bi-level_paper_results",
        "bi_name": "BraninHoo",
    },
    {
        "title": "Hartmann-6",
        "mono_folder": "_experiments/RESS/hartmann_batch_run",
        "mono_name": "ro",
        "bi_folder": "_experiments/RESS/Bi-level_paper_results",
        "bi_name": "Hartmann6",
    },
]

fig, ax = plt.subplots(
    2, 3,
    figsize=1*np.array([300, 180]) / 72.27,
    sharex="col",
    sharey="row",
)

legend_handles = []

for col, problem in enumerate(problems):
    # Mono-level results
    mono_x, mono_q_median, mono_x_median, mono_q_lower, mono_q_upper, mono_x_lower, mono_x_upper = get_mono_level_results(problem["mono_folder"], problem["mono_name"])

    # Bi-level results
    bi_x, bi_q_median, bi_x_median, bi_q_lower, bi_q_upper, bi_x_lower, bi_x_upper = get_rel_error_bi_level(problem["bi_folder"], problem["bi_name"])
    

    # Top row: objective error; bottom row: parameter error.
    for row, (mono_y_median, mono_y_lower, mono_y_upper, bi_y_median, bi_y_lower, bi_y_upper) in enumerate(
        [(mono_q_median, mono_q_lower, mono_q_upper, bi_q_median, bi_q_lower, bi_q_upper), (mono_x_median, mono_x_lower, mono_x_upper, bi_x_median, bi_x_lower, bi_x_upper)]
    ):
        a = ax[row, col]

        mono_line, = a.semilogy(
            mono_x, mono_y_median,
            color=ccycle[0],
            linewidth=1.5,
            linestyle="-",
            label="Mono-level",
        )

        a.fill_between(
            mono_x,
            mono_y_lower.numpy(),
            mono_y_upper.numpy(),
            color=ccycle[0],
            alpha=0.25,
            label="60% central region",
        )


        bi_line, = a.semilogy(
            bi_x, bi_y_median,
            color=ccycle[1],
            linewidth=1.5,
            linestyle="-",
            label="Bi-level",
        )

        a.fill_between(
            bi_x,
            bi_y_lower.numpy(),
            bi_y_upper.numpy(),
            color=ccycle[1],
            alpha=0.25,
            label="60% central region",
        )

        if row == 0 and col == 0:
            legend_handles = [mono_line, bi_line]

        # a.set_xlim(0, 1000)
        a.grid(alpha=0.3)

        # Show tick labels only on the outer axes.
        a.tick_params(axis="x", which="both", labelbottom=(row == 1))
        a.tick_params(axis="y", which="both", labelleft=(col == 0))

        sns.despine(ax=a)

    # ax[0, col].set_title(problem["title"])
    ax[0, col].set_yticks([1, 0.1, 0.01, 0.001])
    ax[1, col].set_yticks([1, 0.1, 0.01, 0.001])
    ax[1, col].set_xlabel("Model calls")

# ax[0, 0].set_ylabel(r"$|q - q^*| / |q^*|$")
# ax[1, 0].set_ylabel(r"$\|x - x^*\|_2 / \|x^*\|_2$")
ax[0, 0].set_ylabel(r"$ \epsilon_{q, \mathrm{rel}}$")
ax[1, 0].set_ylabel(r"$ \epsilon_{x, \mathrm{rel}}$")

fig.legend(
    handles=legend_handles,
    labels=["Mono-level", "Bi-level"],
    loc="upper center",
    bbox_to_anchor=(0.5, 0.985),
    ncol=2,
    frameon=False,
    fontsize=9,
)

fig.text(
    0.15, 0.0, r"\textsf{\textbf{(a)}}",
    ha="left",
    va="bottom",
    fontsize=8,
    color="black",
)

fig.text(
    0.43, 0, r"\textsf{\textbf{(b)}}",
    ha="left",
    va="bottom",
    fontsize=8,
    color="black",
)

fig.text(
    0.72, 0, r"\textsf{\textbf{(c)}}",
    ha="left",
    va="bottom",
    fontsize=8,
    color="black",
)

fig.tight_layout(rect=[0, 0, 1, 0.92])

fig.savefig(
    "_plots/RESS/mono_bi_comparison.pdf",
    bbox_inches="tight",
    pad_inches=0.0,
)




### Compute savings for the different models

0: Sabater <br>
1: Branin <br>
2: Hartmann <br>

In [ ]:
i = 1
mono_x, mono_q_median, mono_x_median, mono_q_lower, mono_q_upper, mono_x_lower, mono_x_upper = get_mono_level_results(problems[i]["mono_folder"], problems[i]["mono_name"])

# Bi-level results
bi_x, bi_q_median, bi_x_median, bi_q_lower, bi_q_upper, bi_x_lower, bi_x_upper = get_rel_error_bi_level(problems[i]["bi_folder"], problems[i]["bi_name"])

In [ ]:
import torch

mono_x_pt = torch.as_tensor(mono_x)
mono_x_error_pt = torch.as_tensor(mono_x_median)


def get_model_eval_to_reach_tol(errors, model_evals, threshold):
    errors = torch.as_tensor(errors)

    # Indices where the tolerance is not satisfied
    above = torch.where(errors > threshold)[0]

    if above.numel() == 0:
        # Error is below the threshold from the beginning
        start_index = 0
    else:
        start_index = int(above[-1]) + 1

    # No suffix satisfies the tolerance
    if start_index >= len(errors):
        return None

    return model_evals[start_index]


thrsholds = np.linspace(0.01, 0.1, 10)
frac = []
for t in thrsholds:
    num_model_mono = get_model_eval_to_reach_tol(
        mono_x_error_pt,
        mono_x_pt,
        t,
    )

    num_model_bi = get_model_eval_to_reach_tol(
            bi_x_median,
            bi_x,
            t,
        )

    if num_model_mono is None:
        print("mono did no reach this")
    elif num_model_bi is None:
        print("bi did no reach this")
    else:
        print(f"{num_model_mono}/{num_model_bi}")
        frac.append(num_model_mono/num_model_bi)

print(frac)
print(f"Saving ratio = {1 - np.mean(np.array(frac))}")